# Import UN Trade and Development (UNCTAD) API data

- Author: Bryan Bravo
- Created: 2026-03-24
- Modified by Bryan Bravo on 2026-04-03: Adjusted time period for imports data to start from 2005 to capture more historical data, as some countries have missing values for 2006. Also added filtering to include only USD values for FX reserves, as there are additional rows with unit measure 'XDR' that are not relevant for our analysis.
## Import Libraries

In [1]:
import os
import sys
os.chdir("C:/Users/bravo/OneDrive/bravo_projects/MLProject/StraitofHormuzResearch")

# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)


# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

import proj_vars

### Initialize Spark Session


In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


### Custom Functions

### Variables

In [3]:
end_date = (dt.now().replace(day=1) - relativedelta(days=1)).strftime("%Y-%m-%d")
out_path = 's3a://ml-project-s3-bronze/input_folder/'
end_date[:7]

'2026-03'

## Query

In [4]:
country_mapping = {
    'australia': 'AUS',
    'brazil': 'BRA',
    'canada': 'CAN',
    'china': 'CHN',
    # 'euro': 'EU',  # No IMF Data, must source from elsewhere.
    'france': 'FRA',
    'germany': 'DEU',
    'india': 'IND',
    'italy': 'ITA',
    'japan': 'JPN',
    'mexico': 'MEX',
    'south_korea': 'KOR',
    'russia': 'RUS',
    'south_africa': 'ZAF',
    'turkiye': 'TUR',
    'united_kingdom': 'GBR',
    'united_states': 'USA'
}


### Import FX monthly reserves

In [5]:
def import_wb_fx_reserves(country_code, end_date):
    response = requests.get(
        f"https://data360api.worldbank.org/data360/data?DATABASE_ID=IMF_IFS&INDICATOR=IMF_IFS_RAXG&REF_AREA={country_code}" +
        f"&FREQ=M&timePeriodFrom=2006-01&timePeriodTo={end_date[:7]}&skip=0"
    )
    wb_data = response.json()

    if 'error_code' in wb_data:  # check if response is None
        print(f"✗ WB API error: {wb_data.get('error_code', 'error_message')}")

    wb_df = pd.DataFrame(wb_data['value'])
    wb_df.columns = [col.lower() for col in wb_df.columns]
    ## Corrected on 04-03-2026: 
        # There are additional rows where the unit measure has ['USD', 'XDR'], filtering to include only USD values.
    wb_df = wb_df[wb_df['unit_measure'] == 'USD'] 
    wb_df = wb_df[['obs_value', 'ref_area', 'time_period']]
    return wb_df

In [6]:
fx_df = pd.DataFrame(columns=['obs_value', 'ref_area', 'time_period'])

for country_code in country_mapping.values():
    print(f"Importing {country_code} from World Bank Data")
    fx_df = pd.concat([fx_df, import_wb_fx_reserves(country_code, end_date)], ignore_index=True)

fx_df[['year', 'month']] = fx_df['time_period'].str.split('-', expand=True).astype(int)
fx_df['country'] = fx_df['ref_area'].map({code: name for name, code in country_mapping.items()})
fx_df['fx_reserves'] = fx_df['obs_value'].astype(float)

fx_df = fx_df[['country', 'year', 'month', 'fx_reserves']]

Importing AUS from World Bank Data
Importing BRA from World Bank Data
Importing CAN from World Bank Data
Importing CHN from World Bank Data
Importing FRA from World Bank Data
Importing DEU from World Bank Data
Importing IND from World Bank Data
Importing ITA from World Bank Data
Importing JPN from World Bank Data
Importing MEX from World Bank Data
Importing KOR from World Bank Data
Importing RUS from World Bank Data
Importing ZAF from World Bank Data
Importing TUR from World Bank Data
Importing GBR from World Bank Data
Importing USA from World Bank Data


### Import `Imports of goods and services (current US$)`

In [8]:
# https://data360.worldbank.org/en/indicator/WB_WDI_NE_IMP_GNFS_CD
def import_wb_imports_dol(country_code, end_date):
    response = requests.get(
        f"https://data360api.worldbank.org/data360/data?DATABASE_ID=WB_WDI&INDICATOR=WB_WDI_NE_IMP_GNFS_CD&REF_AREA={country_code}" +
        f"&timePeriodFrom=2005-01&timePeriodTo={end_date[:4]}&skip=0"  # Corrected on 04-03-2026: Adjusted time period to start from 2005 to capture more historical data, as some countries have missing values for 2006.
    )
    wb_data = response.json()

    if 'error_code' in wb_data:  # check if response is None
        print(f"✗ WB API error: {wb_data.get('error_code', 'error_message')}")

    wb_df = pd.DataFrame(wb_data['value'])
    wb_df.columns = [col.lower() for col in wb_df.columns]
    wb_df = wb_df[['obs_value', 'ref_area', 'time_period']]
    return wb_df

In [9]:
import_df = pd.DataFrame(columns=['obs_value', 'ref_area', 'time_period'])

for country_code in country_mapping.values():
    print(f"Importing {country_code} from World Bank Data")
    import_df = pd.concat([import_df, import_wb_imports_dol(country_code, end_date)], ignore_index=True)
    print(f"Finished importing {country_code}")

import_df['year'] = import_df['time_period'].astype(int)
import_df['country'] = import_df['ref_area'].map({code: name for name, code in country_mapping.items()})
import_df['imports_good_service'] = import_df['obs_value'].astype(float) / 12  # Computes approximate month from annual

import_df = import_df[['country', 'year', 'imports_good_service']]

Importing AUS from World Bank Data
Finished importing AUS
Importing BRA from World Bank Data
Finished importing BRA
Importing CAN from World Bank Data
Finished importing CAN
Importing CHN from World Bank Data
Finished importing CHN
Importing FRA from World Bank Data
Finished importing FRA
Importing DEU from World Bank Data
Finished importing DEU
Importing IND from World Bank Data
Finished importing IND
Importing ITA from World Bank Data
Finished importing ITA
Importing JPN from World Bank Data
Finished importing JPN
Importing MEX from World Bank Data
Finished importing MEX
Importing KOR from World Bank Data
Finished importing KOR
Importing RUS from World Bank Data
Finished importing RUS
Importing ZAF from World Bank Data
Finished importing ZAF
Importing TUR from World Bank Data
Finished importing TUR
Importing GBR from World Bank Data
Finished importing GBR
Importing USA from World Bank Data
Finished importing USA


### Join DFs

In [10]:
wb_df = spark.createDataFrame(
    fx_df.merge(import_df, how='outer', on=['country', 'year'])
    )

wb_df = (
    wb_df
    .withColumn('imports_good_service',  # Making correction on how nulls are recorded.
                F.when(F.col('imports_good_service').contains("NaN"), F.lit(None)).otherwise(F.col('imports_good_service'))
    )
)

In [11]:
wb_df.toPandas().to_csv('processed_datasets/wb.csv', index=False)
wb_df.show()

+---------+----+-----+----------------+--------------------+
|  country|year|month|     fx_reserves|imports_good_service|
+---------+----+-----+----------------+--------------------+
|australia|2006|    8|49523.2349796885|1.353756893915684...|
|australia|2006|    3|42835.1902874583|1.353756893915684...|
|australia|2006|   10|48545.2614477324|1.353756893915684...|
|australia|2006|    2|40007.4980160947|1.353756893915684...|
|australia|2006|   12|53448.1459577347|1.353756893915684...|
|australia|2006|    1|42263.5916751887|1.353756893915684...|
|australia|2006|    6|45866.1904852359|1.353756893915684...|
|australia|2006|   11|49675.4748707124|1.353756893915684...|
|australia|2006|    5| 49615.006092537|1.353756893915684...|
|australia|2006|    4|46839.2052678739|1.353756893915684...|
|australia|2006|    7|50781.7669379911|1.353756893915684...|
|australia|2006|    9|44651.4131142197|1.353756893915684...|
|australia|2007|    2|52027.7823160236|1.555010337337415...|
|australia|2007|    8|56